<div dir="rtl" style="text-align:right; line-height:1.9; font-size:17px">\n<h1>📦 Smart Compressor</h1>\n<p><b>الاستخدام العادي بسيط:</b></p>\n<p>١) ابعت ملفات الصوت أو الفيديو في قناة <b>📦 Smart Compressor</b> على تيليجرام.<br>\n٢) ارجع هنا واضغط زر التشغيل ▶.<br>\n٣) النتيجة هترجع لنفس القناة تلقائيًا.</p>\n<p><b>عايز تعالج نتيجة قديمة مرة تانية؟</b><br>\nاعمل رد على الملف في تيليجرام واكتب أحد الأوامر دي:</p>\n<p>🔁 لإعادة المعالجة بنفس الوضع الحالي<br>\n🔁 أصغر لضغط الفيديو بشكل أقوى<br>\n🔁 صوت لاستخراج صوت صغير جدًا<br>\n🔁 80 لاستهداف حجم يقارب 80 ميجابايت</p>\n</div>

In [ ]:
#@title ▶ تشغيل Smart Compressor\n#@markdown اختار الوضع فقط لو محتاج تغير الإعداد الافتراضي.\nالوضع = "تلقائي — مناسب لمعظم الاستخدامات" #@param ["تلقائي — مناسب لمعظم الاستخدامات", "صوت صغير جدًا", "فيديو متوازن", "فيديو سريع", "أصغر حجم للفيديو", "حجم فيديو محدد"]\nالحجم = "" #@param {type:"string"}\n\nimport os, re, subprocess, urllib.request, json, time\n\nREPO = "abdullahsamirashour/gpt"\nBRANCH = "main"\nENGINE_REL = "telegram-smart-compressor/engine.py"\nENGINE_PATH = "/content/telegram_smart_compressor_engine.py"\n\nos.environ["TSC_PROFILE"] = str(الوضع or "تلقائي — مناسب لمعظم الاستخدامات")\nos.environ["TSC_TARGET_SIZE_MB"] = str(الحجم or "").strip()\nos.environ["TSC_UPDATE_CHANNEL"] = BRANCH\n\ndef latest_sha():\n    p = subprocess.run(["git", "ls-remote", f"https://github.com/{REPO}.git", f"refs/heads/{BRANCH}"], capture_output=True, text=True, timeout=30)\n    if p.returncode == 0 and p.stdout.strip():\n        sha = p.stdout.strip().split()[0]\n        if re.fullmatch(r"[0-9a-f]{40}", sha):\n            return sha\n    req = urllib.request.Request(\n        f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",\n        headers={"Accept":"application/vnd.github+json", "User-Agent":"Smart-Compressor-Colab", "Cache-Control":"no-cache"},\n    )\n    with urllib.request.urlopen(req, timeout=30) as r:\n        return json.loads(r.read().decode("utf-8"))["object"]["sha"]\n\ndef download_engine(sha):\n    url = f"https://raw.githubusercontent.com/{REPO}/{sha}/{ENGINE_REL}"\n    req = urllib.request.Request(url, headers={"User-Agent":"Smart-Compressor-Colab", "Cache-Control":"no-cache"})\n    with urllib.request.urlopen(req, timeout=30) as r:\n        return r.read().decode("utf-8")\n\ntry:\n    print("🔄 جاري تحميل أحدث نسخة مستقرة...")\n    sha = latest_sha()\n    engine = download_engine(sha)\n    m = re.search(r'ENGINE_BUNDLE_VERSION\\s*=\\s*"([^"]+)"', engine)\n    if len(engine) < 5000 or not m:\n        raise RuntimeError("invalid-engine")\n    print(f"✅ النسخة جاهزة: {m.group(1)}")\n    with open(ENGINE_PATH, "w", encoding="utf-8") as f:\n        f.write(engine)\n    exec(compile(engine, ENGINE_PATH, "exec"), globals(), globals())\nexcept SyntaxError:\n    print("❌ E002")\n    print("النسخة المحمّلة فيها خطأ برمجي. ابعت كود الخطأ E002.")\nexcept Exception as e:\n    print("❌ E001")\n    print("تعذر تحميل المحرك من GitHub. تأكد من اتصال الإنترنت وجرب مرة أخرى.")\n